# cop-fx-intelligence — El producto end-to-end (sistema REAL)

> Esta notebook corrió antes como *walking skeleton* (heurísticas + datos sintéticos).
> Ese esqueleto ya cumplió su función: **hoy ejecuta el sistema de producción** —
> el grafo LangGraph completo con datos y LLM reales.

**El foco del proyecto** (no ha cambiado nunca): clasificar la **dirección** del USD/COP
(`down` / `up` / `neutral`), no la magnitud, cruzando dos señales independientes —
noticias analizadas por agentes y el signo del forecast — con una capa de racionalidad
que obliga al contra-argumento y acota la confianza **por contrato Pydantic**.

| Etapa | Qué es | Dónde vive |
|---|---|---|
| 0 | Contratos Pydantic (la racionalidad vive aquí) | `src/cop_fx/contracts.py` |
| 1 | Análisis estructurado (`with_structured_output`) | `src/cop_fx/analysis/news_analyzer.py` |
| 2 | Router de materialidad (gate de costo) | `nodes.check_materiality` |
| 3 | Orchestrator-workers con `Send` (fan-out por cluster) | `nodes.orchestrate` / `topic_worker` |
| 4 | Adjudicador (reconciliación + abogado del diablo) | `nodes.adjudicate` |
| 5 | Tabla `predictions` + backtest direccional | `src/cop_fx/tracking/` |

**El producto diario NO vive en las notebooks**: vive en `cop-fx run` → `reports/report_YYYY-MM-DD.md`
y en el dashboard (`uv run streamlit run dashboard/app.py`). Las notebooks son el laboratorio.

In [1]:
# DEBE IR PRIMERO — diagnóstico de entorno + carga del .env
import os
import sys
from pathlib import Path

# 1) ¿Kernel correcto? El paquete del proyecto debe ser importable.
try:
    import cop_fx  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(
        f"KERNEL EQUIVOCADO: este Python ({sys.executable}) no tiene el paquete cop_fx.\n"
        "En VS Code: clic en el selector de kernel (arriba a la derecha) y elige "
        "'Python (cop-fx-intelligence)'."
    ) from None

# 2) Raíz del repo (funciona corras desde donde corras) y .env con fallback.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
os.chdir(ROOT)  # data/, reports/ y el .env del pipeline son relativos a la raíz

from dotenv import load_dotenv

for env_file in (ROOT / ".env", ROOT / "notebooks" / ".env"):
    if env_file.exists():
        load_dotenv(env_file, override=False)

api_key = os.getenv("OPENAI_API_KEY", "")
if len(api_key) < 40 or api_key.startswith("sk-ant"):
    raise EnvironmentError(
        "OPENAI_API_KEY no encontrada o es un placeholder.\n"
        f"Busqué en: {ROOT / '.env'} y {ROOT / 'notebooks' / '.env'}\n"
        "Agrega tu clave real de OpenAI a uno de esos archivos."
    )

print(f"✓ Kernel  : {sys.executable}")
print(f"✓ Repo    : {ROOT}")
print(f"✓ API key : {api_key[:12]}…")

✓ Kernel  : /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/bin/python3
✓ Repo    : /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence
✓ API key : sk-proj-s76m…


## 1. El grafo de inteligencia

Este es el sistema completo compilado. Léelo de izquierda a derecha:
dos ramas paralelas (FX y noticias), el **router** que decide si el día amerita gastar tokens,
el **fan-out dinámico** (`Send`) con un worker por cluster de tópico, la convergencia en el
**adjudicador** (`defer=True`, un solo trigger — ver el docstring de `build_graph` para el
porqué), y al final reporte → tracking → publicación.

In [2]:
from cop_fx.agents.graph import build_graph

compiled = build_graph().compile()
print(compiled.get_graph().draw_mermaid())

Importing plotly failed. Interactive plots will not work.


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	fetch_fx(fetch_fx)
	fetch_market(fetch_market)
	fetch_news(fetch_news)
	check_materiality(check_materiality)
	skip_news(skip_news)
	orchestrate(orchestrate)
	topic_worker(topic_worker)
	aggregate_signals(aggregate_signals)
	run_forecast(run_forecast)
	adjudicate(adjudicate<hr/><small><em>defer = True</em></small>)
	generate_report(generate_report)
	record_prediction(record_prediction)
	publish(publish)
	__end__([<p>__end__</p>]):::last
	__start__ --> fetch_fx;
	__start__ --> fetch_market;
	__start__ --> fetch_news;
	adjudicate --> generate_report;
	aggregate_signals --> adjudicate;
	check_materiality -.-> orchestrate;
	check_materiality -.-> skip_news;
	fetch_fx --> run_forecast;
	fetch_news --> check_materiality;
	generate_report --> record_prediction;
	orchestrate -.-> topic_worker;
	record_prediction --> publish;
	skip_news --> adjudicate;
	topic_worker --> aggregate_signals;
	fetch_mark

## 2. Una corrida real completa

Lo mismo que ejecuta `uv run cop-fx run` (y el cron diario de GitHub Actions).
Descarga la TRM oficial (datos.gov.co), scrapea CNN + feeds macro, y gasta
~8-10 llamadas a `gpt-5.4-mini` (≈ fracciones de centavo). Tarda ~40-60 s.

In [3]:
from cop_fx.agents.graph import run_pipeline

final_state = run_pipeline()

print("\n=== resumen de la corrida ===")
print(f"TRM actual        : {final_state.get('latest_rate', 0):,.2f} ({final_state.get('rate_change_pct', 0):+.2f}% 30d)")
print(f"Artículos crudos  : {len(final_state.get('raw_articles', []))}")
print(f"¿Día material?    : {final_state.get('has_material_news')}")
print(f"Clusters (Send)   : { {t: len(i) for t, i in final_state.get('clusters', {}).items()} }")
print(f"Errores           : {final_state.get('errors', [])}")

15:21:22  INFO      cop_fx.data.fx_fetcher  —  FX source: TRM datos.gov.co


15:21:22  INFO      cop_fx.data.fx_fetcher  —  Fetched 236 FX rows (start=2025-06-12, end=2026-06-12)


15:21:23  INFO      cop_fx.agents.nodes  —  Market context: down (equity +5.48%, dxy -0.08%, brent -3.78%)


15:21:23  INFO      cop_fx.data.cnn_fetcher  —  CNNColombiaFetcher: starting (max=50)


15:21:24  WARNING   cop_fx.data.cnn_fetcher  —  RSS feed returned 0 entries: https://cnnespanol.cnn.com/colombia/feed/


...

15:21:33  INFO      cop_fx.agents.nodes  —  Señal de noticias: neutral (score=-0.30, drivers=0)


15:21:38  INFO      cop_fx.agents.nodes  —  DirectionalCall: down (confianza=0.43, reconciliación=partial)


15:21:38  INFO      cop_fx.tracking.predictions  —  Prediction saved: 2026-06-12 → down (0.43)


15:21:38  INFO      cop_fx.agents.nodes  —  publish: skipped (publish_enabled=False)



=== resumen de la corrida ===
TRM actual        : 3,513.54 (-2.22% 30d)
Artículos crudos  : 50
¿Día material?    : True
Clusters (Send)   : {'political_risk': 5, 'energy_commodities': 2, 'financial_markets': 2, 'us_global_macro': 2, 'monetary_policy': 2, 'other': 2}
Errores           : []


## 3. El producto: `DirectionalCall`

El veredicto del adjudicador. Fíjate en tres cosas que NO son cosmética:

1. **`reconciliation`** se declara antes de decidir — si las señales divergen, la confianza
   tiene techo 0.5 *impuesto por el validador del contrato*, no por cortesía del prompt.
2. **`devils_advocate`** es obligatorio (mín. 20 caracteres): el modelo no puede emitir
   un veredicto sin construir el mejor argumento en contra.
3. **`neutral` es una salida válida**: confianza < 0.35 ⇒ el sistema se abstiene.

In [4]:
call = final_state["directional_call"]

ARROW = {"down": "⬇️ USD/COP BAJA (COP se fortalece)",
         "up": "⬆️ USD/COP SUBE (COP se debilita)",
         "neutral": "⏸️ NEUTRAL — abstención"}

print("=" * 70)
print(f"  {ARROW[call['direction']]}")
print(f"  confianza {call['confidence']:.2f} · horizonte {call['horizon_days']}d · reconciliación: {call['reconciliation']}")
print("=" * 70)
print(f"  noticias: {call['news_signal']['direction']} (score {call['news_signal']['score']})")
print(f"  serie   : {call['ts_signal']['direction']} ({call['ts_signal']['yhat_delta_pct']:+.2f}%, modelos {'concuerdan' if call['ts_signal']['models_agree'] else 'difieren'})")
print("-" * 70)
print(f"RACIONAL:\n{call['rationale']}\n")
print(f"ABOGADO DEL DIABLO:\n{call['devils_advocate']}\n")
print("CAVEATS:")
for c in call["caveats"]:
    print(f"  - {c}")

  ⬇️ USD/COP BAJA (COP se fortalece)
  confianza 0.43 · horizonte 7d · reconciliación: partial
  noticias: neutral (score -0.3)
  serie   : down (-0.45%, modelos difieren)
----------------------------------------------------------------------
RACIONAL:
Las señales son parcialmente divergentes: el componente de noticias está casi neutro con sesgo leve a favor del COP (score -0.3, es decir, presión muy pequeña a la baja en USD/COP) por la caída del petróleo, algo de mejora social/macro y un leve soporte por crecimiento global; pero también hay ruido político local, movilizaciones en Ecopetrol y un sesgo mixto externo que limitan la convicción. El signal de serie temporal sí es claramente bajista para USD/COP (yhat_delta_pct -0.447, dirección down), y además está respaldado por el contexto de mercado: el agregado accionario colombiano subió fuerte ayer (+5.485%), que es el único leading indicator validado y suele anticipar COP más fuerte / USD-COP a la baja. Dado que la noticia es de baja

## 4. El reporte diario

Lo que el sistema persiste en `reports/` (y tuitearía con `--publish`).

In [5]:
from IPython.display import Markdown, display

display(Markdown(final_state["report_markdown"]))

# COP/USD Intelligence Report — 2026-06-12

## Current Rate
**1 USD = 3,513.54 COP** 📉 (-2.22% vs 30 days ago)

## Directional Call — 7 días
**⬇️ USD/COP BAJA (COP se fortalece)** · confianza **0.43** · reconciliación **partial**

Señales: noticias = neutral (score -0.3) · serie = down (-0.45%, modelos difieren)

Mercado (ayer): bolsa CO +5.49% · DXY -0.08% · Brent -3.78% → sesgo down

**Racional:** Las señales son parcialmente divergentes: el componente de noticias está casi neutro con sesgo leve a favor del COP (score -0.3, es decir, presión muy pequeña a la baja en USD/COP) por la caída del petróleo, algo de mejora social/macro y un leve soporte por crecimiento global; pero también hay ruido político local, movilizaciones en Ecopetrol y un sesgo mixto externo que limitan la convicción. El signal de serie temporal sí es claramente bajista para USD/COP (yhat_delta_pct -0.447, dirección down), y además está respaldado por el contexto de mercado: el agregado accionario colombiano subió fuerte ayer (+5.485%), que es el único leading indicator validado y suele anticipar COP más fuerte / USD-COP a la baja. Dado que la noticia es de baja intensidad y el marco de mercado favorece la dirección técnica, domina el forecast temporal. Por eso el veredicto agregado es DOWN para los próximos 7 días, pero con convicción moderada y no alta.

**Abogado del diablo:** El mejor caso contra DOWN es que el lote de noticias no muestra un catalizador claro de fortaleza del COP: el riesgo político local, la incertidumbre por Ecopetrol y el sesgo global mixto pueden pesar más que la caída del petróleo, especialmente si el mercado ya descontó parte del movimiento accionario de ayer. Además, los modelos de serie temporal no están completamente alineados (models_agree=false), lo que sugiere fragilidad del pronóstico. Si el rebote de equities colombianas fue puntual y no se sostiene, el COP podría perder impulso y el USD/COP terminar plano o incluso subir ligeramente.

**Drivers:** (sin drivers de alta severidad)

**Caveats:**
- El sesgo de noticias es bajo en magnitud y mezcla horizontes distintos; varios artículos parecen de impacto limitado o diferido.
- Solo el agregado accionario colombiano tiene validación empírica fuerte como indicador líder; DXY y Brent se usan solo como contexto.
- Los modelos de time series no están totalmente de acuerdo, lo que reduce la robustez del forecast.
- Puede haber ruido por titulares políticos locales y por la situación de Ecopetrol, difícil de cuantificar en 7 días.
- La confianza está acotada por la divergencia parcial y por la intensidad moderada de los drivers.

## Market Narrative
[political_risk] La jornada luce dominada por titulares de riesgo político local, con el caso Petro como principal foco de incertidumbre. El canal relevante es country risk, aunque la mayoría de notas sugieren impactos solo moderados o bajos sobre el USD/COP. En conjunto, el sesgo para el COP es ligeramente negativo por mayor ruido electoral y diplomático.
[energy_commodities] La caída del petróleo a mínimos de tres meses es el principal factor FX del lote y tiende a favorecer al COP vía términos de intercambio. El plan del Caribe para convertirse en hub energético y exportador de energía limpia también apunta, de forma más lejana, a una mejora estructural de los términos de intercambio. En conjunto, el sesgo diario es moderadamente favorable para el COP, aunque el segundo artículo tiene impacto limitado por su horizonte de ejecución.
[financial_markets] Las noticias apuntan a una mejora gradual de las condiciones sociales en Colombia, lo que sugiere algo más de fortaleza macro. El canal relevante es de crecimiento, con efecto pequeño sobre el USD/COP. En conjunto, el sesgo para el COP es ligeramente positivo, pero de baja intensidad.
[us_global_macro] Las noticias apuntan a un sesgo mixto para el FX global: la debilidad del Reino Unido sugiere menor crecimiento, lo que podría suavizar al dólar marginalmente. En contraste, el repunte de precios al productor por energía refuerza presiones inflacionarias y apoyo al dólar vía expectativas de tasas. Para USD/COP, el balance luce de impacto bajo y ligeramente alcista para el COP solo por el canal de crecimiento global.
[monetary_policy] Las noticias apuntan a un sesgo ligeramente alcista para el USD por el deterioro del crecimiento y las presiones inflacionarias en Europa. El canal principal es indirecto: mayor incertidumbre sobre crecimiento global y tasas más restrictivas en la zona euro. Para COP, el efecto sería limitado pero levemente negativo vía menor apetito por riesgo.
[other] Las noticias apuntan a un tono levemente negativo para el COP por el lado de riesgo local y actividad. Las movilizaciones en Ecopetrol podrían elevar la incertidumbre sobre producción y conflicto laboral, aunque el impacto esperado es limitado. El aumento proyectado de cáncer añade una presión estructural sobre crecimiento y gasto público, pero sin choque inmediato para el tipo de cambio.

## 7-Day Forecast
| Date | Forecast | 95% CI |
|------|----------|--------|
| 2026-06-15 | 3536.71 | 3469.64 – 3589.10 |
| 2026-06-16 | 3518.32 | 3450.53 – 3594.17 |
| 2026-06-17 | 3517.60 | 3428.82 – 3603.91 |
| 2026-06-18 | 3513.08 | 3415.54 – 3614.98 |
| 2026-06-19 | 3513.50 | 3409.12 – 3633.04 |
| 2026-06-22 | 3521.68 | 3396.48 – 3643.19 |
| 2026-06-23 | 3497.83 | 3382.49 – 3647.65 |

## Model Performance (back-test)
- **PROPHET**: MAE=59.06, RMSE=72.22, MAPE=1.65%
- **ARIMA**: MAE=8.87, RMSE=10.15, MAPE=0.25%


---
*Generated by cop-fx-intelligence at 2026-06-12T20:21:38Z*


## 5. Etapa 5 — cerrar el loop: `predictions` + backtest

La corrida de arriba ya guardó su predicción en `data/predictions.db` (nodo
`record_prediction`). Cada corrida futura **evalúa sola** las predicciones cuyo
horizonte venció, comparando contra la TRM real. Con el tiempo esto responde LA
pregunta: ¿el sistema le gana a un baseline trivial? ¿Y acierta más cuando dice
"alta confianza"? (eso valida la calibración del adjudicador).

In [6]:
from cop_fx.tracking import PredictionStore

store = PredictionStore("data/predictions.db")
preds = store.all()
print(f"{len(preds)} predicciones registradas")
preds[["run_date", "direction", "confidence", "reconciliation",
       "news_direction", "ts_direction", "latest_rate", "actual_direction", "hit"]]

1 predicciones registradas


,run_date,direction,confidence,reconciliation,news_direction,ts_direction,latest_rate,actual_direction,hit
0,2026-06-12,down,0.43,partial,neutral,down,3513.54,None,None


### Backtest de la señal de serie — la prueba de honestidad (sin LLM)

Mientras la tabla `predictions` acumula historia real, podemos backtestear YA la
mitad determinista del sistema: en cada uno de los últimos N días, ¿el signo de
ARIMA predijo la dirección a 5 días? ¿Le gana a `momentum` (repetir el último
movimiento) y a `always_up`? **Si no les gana, la serie no aporta y todo el peso
del sistema recae en las noticias** — eso es exactamente lo que queremos saber.

In [7]:
import pandas as pd

from cop_fx.tracking import directional_backtest

detail, summary = directional_backtest(final_state["fx_df"], horizon_days=5, n_origins=40)

print(f"Backtest: {summary['n_origins']} orígenes, horizonte {summary['horizon_days']}d\n")
rows = []
for strategy in ("arima", "momentum", "always_up"):
    s = summary[strategy]
    rows.append({
        "estrategia": strategy,
        "hit_rate": f"{s['hit_rate']:.0%}" if s["hit_rate"] is not None else "—",
        "decisiones": s["n_decided"],
        "abstenciones": s["n_abstained"],
    })
pd.DataFrame(rows)

15:21:38  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3664.41, day+5=3670.42


15:21:38  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3678.19, day+5=3682.31


15:21:38  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3640.63, day+5=3640.30


15:21:38  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3642.93, day+5=3636.56


/Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

15:21:40  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3560.24, day+5=3559.13


15:21:40  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3562.00, day+5=3572.94


15:21:40  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3572.85, day+5=3567.83


Backtest: 40 orígenes, horizonte 5d



,estrategia,hit_rate,decisiones,abstenciones
0,arima,52%,25,15
1,momentum,53%,36,4
2,always_up,42%,40,0


In [8]:
# ¿Cómo se distribuyen los aciertos en el tiempo?
detail["arima_hit"] = (detail["arima"] == detail["actual"]) & (detail["arima"] != "neutral")
print(detail.tail(10).to_string(index=False))

    origin actual   arima momentum always_up  arima_hit
2026-05-21   down neutral     down        up      False
2026-05-22   down neutral     down        up      False
2026-05-23   down      up     down        up      False
2026-05-27   down neutral     down        up      False
2026-05-28   down      up     down        up      False
2026-05-29   down neutral       up        up      False
2026-05-30   down    down       up        up       True
2026-06-02     up neutral     down        up      False
2026-06-03     up      up  neutral        up       True
2026-06-04   down    down       up        up       True


---
# Conclusiones del ejercicio — de principio a fin

## ¿Qué hace cada agente? (el reparto de trabajo)

| # | Agente / nodo | Qué hace | Patrón agéntico | ¿LLM? |
|---|---|---|---|---|
| 1 | `fetch_fx` | Baja la TRM oficial (datos.gov.co) | — | No |
| 2 | `fetch_news` | Scrapea CNN Colombia + feeds macro (CNBC, MarketWatch) | — | No |
| 3 | `check_materiality` | **El portero**: lee SOLO los titulares y decide si el día puede mover el dólar. De paso etiqueta cada titular con un tópico grueso | Routing (gate de costo) | 1 llamada barata |
| 4 | `skip_news` | Día sin señal → ruta gratis, el forecast carga solo | — | No |
| 5 | `orchestrate` | Agrupa los titulares materiales en clusters por tópico (usa las etiquetas que el portero ya pagó) | Orchestrator | No |
| 6 | `topic_worker` × N | **Los analistas**: un agente POR CLUSTER analiza a fondo solo sus artículos — tópico fino, entidades, canal de transmisión FX, severidad, dirección. N lo decide el dato del día, no el código | `Send` fan-out + prompt chaining | N llamadas en paralelo |
| 7 | `aggregate_signals` | Suma los veredictos en UNA señal (peso = severidad × relevancia; deportes pesa 0 por contrato) | Reducer determinista | No |
| 8 | `run_forecast` | Prophet + ARIMA(2,1,2) → el SIGNO del ensemble = señal de la serie | — | No |
| 9 | `adjudicate` | **El juez**: cruza noticias vs serie, declara si concuerdan o divergen, construye el abogado del diablo y emite el veredicto. Sus números no son suyos: las señales se las inyecta el sistema y los validadores Pydantic le acotan la confianza | Evaluator / racionalidad | 1 llamada (tier judge) |
| 10 | `record_prediction` | Guarda el veredicto y califica las predicciones pasadas cuyo horizonte venció | Tracking | No |
| 11 | `generate_report` / `publish` | Markdown a `reports/` y tweet opcional | — | No |

**La división clave**: los LLM solo razonan sobre TEXTO (¿qué significa esta noticia?);
todos los NÚMEROS (señales, pesos, confianza acotada, evaluación) los produce código
determinista. El LLM nunca puede inventarse un score.

## ¿Qué demostró el ejercicio? (los resultados, honestos)

1. **El gate funciona**: en días de solo deportes la rama de análisis ni se ejecuta
   (cero tokens); en días mixtos descarta los titulares irrelevantes antes de pagar.
2. **El fan-out es real**: la cantidad de analistas la decide el día (hoy se verán los
   clusters arriba) — eso es lo que `Send` permite y un grafo estático no.
3. **La serie sola NO predice**: el backtest dio ~52% para ARIMA(2,1,2) ≈ momentum ≈
   moneda (y el diagnóstico ACF/PACF de `series_de_tiempo.ipynb` explica por qué:
   los retornos casi no tienen memoria lineal). **Esto valida el diseño**: si la serie
   bastara, los agentes sobrarían.
4. **La racionalidad se nota en el output**: el veredicto de hoy declara su divergencia,
   su confianza queda topada bajo 0.5 y el abogado del diablo es una objeción real.
   Compara esto con pedirle a ChatGPT "¿sube el dólar?" — esa es la diferencia entre
   un sistema y un prompt.
5. **El sistema ya es medible**: cada corrida deja una fila en `predictions` que se
   auto-califica al vencer su horizonte. En ~30 corridas diarias habrá hit-rate real
   del sistema completo (noticias + serie + juez), que es el número que decidirá si
   `gpt-5.4-mini` basta como juez o hay que subir de modelo.

In [9]:
# Conclusión GENERADA de esta corrida (se reescribe cada vez que ejecutas la notebook)
call = final_state["directional_call"]
ns, ts = call["news_signal"], call["ts_signal"]
n_workers = len(final_state.get("clusters", {}))
n_articles = len(final_state.get("raw_articles", []))
n_analyzed = len(final_state.get("analyzed_articles", []))

print(f"""
┌─ CONCLUSIÓN DE LA CORRIDA ─────────────────────────────────────────
│ De {n_articles} titulares, el portero juzgó el día {'MATERIAL' if final_state.get('has_material_news') else 'NO material'}
│ y despachó {n_workers} analistas en paralelo que examinaron {n_analyzed} artículos.
│
│ Señal de NOTICIAS : {ns['direction']:>7}  (score {ns['score']:+.2f})
│ Señal de la SERIE : {ts['direction']:>7}  (Δ forecast {ts['yhat_delta_pct']:+.2f}%)
│ El juez declaró   : {call['reconciliation'].upper()}
│
│ VEREDICTO: USD/COP {call['direction'].upper()} a {call['horizon_days']} días, confianza {call['confidence']:.2f}
│ {'(confianza topada en 0.5 por divergencia — regla del contrato)' if call['reconciliation'] == 'diverge' and call['confidence'] == 0.5 else ''}
│ Registrado en data/predictions.db — se autocalificará al vencer el horizonte.
└────────────────────────────────────────────────────────────────────
""")


┌─ CONCLUSIÓN DE LA CORRIDA ─────────────────────────────────────────
│ De 50 titulares, el portero juzgó el día MATERIAL
│ y despachó 6 analistas en paralelo que examinaron 15 artículos.
│
│ Señal de NOTICIAS : neutral  (score -0.30)
│ Señal de la SERIE :    down  (Δ forecast -0.45%)
│ El juez declaró   : PARTIAL
│
│ VEREDICTO: USD/COP DOWN a 7 días, confianza 0.43
│ 
│ Registrado en data/predictions.db — se autocalificará al vencer el horizonte.
└────────────────────────────────────────────────────────────────────



---
## Qué sigue

- **Etapa 6**: checkpointer (`SqliteSaver`), `interrupt` antes de publicar (HITL: tú apruebas
  el tweet) y memoria entre corridas ("ayer dije down con 0.7 y fallé" como contexto del
  adjudicador). Curso: módulos 06/07/12.
- **GCP** (fases 5-6 de `docs/arquitectura.md`): Cloud Run Jobs + Scheduler + BigQuery —
  swaps de adapter, nada de lo de arriba se reescribe.

**Para replicar y dominar LangGraph**: reconstruye este grafo desde cero en una notebook
vacía, en este orden — (1) un nodo con `with_structured_output`, (2) el router con
`add_conditional_edges`, (3) el fan-out con `Send` y reducers `operator.add`,
(4) el join con `defer=True` y UN solo trigger. Cada paso tiene su test en
`tests/unit/test_graph_nodes.py` para verificarte.